# 03 — End-to-End Demo

Full system walkthrough without launching any servers — runs the pipeline directly in Python:

1. IntentDetector — classify various inputs
2. MoodExtractor  — Gemini Call 1 with real API
3. FAISSRetriever — semantic search
4. RetrievalEvaluator — quality gating
5. GeminiGenerator — Gemini Call 2
6. Full pipeline — single call

> **Prerequisite:** FAISS index built (notebook 02) and `.env` populated.

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded.')

## 1. IntentDetector — classify all edge-case inputs

In [ ]:
from backend.agent.intent_detector import IntentDetector

detector = IntentDetector()

test_inputs = [
    ('I feel melancholic',     'interpretable'),
    ('idk',                    'interpretable'),
    ('🥺',                     'interpretable'),
    ('aaaaaa',                 'interpretable'),
    ('burnt out',              'interpretable'),
    ('like Dark but easier',   'interpretable'),
    ('something good',         'interpretable'),
    ('',                       'empty'),
    ('   ',                    'empty'),
]

print(f'{"Input":<30} {"Expected":<14} {"Got":<14} {"Pass"}')
print('-' * 65)
all_pass = True
for text, expected in test_inputs:
    got = detector.classify(text)
    passed = got == expected
    all_pass = all_pass and passed
    marker = '✓' if passed else '✗'
    print(f'{repr(text):<30} {expected:<14} {got:<14} {marker}')

print(f'\nAll tests passed: {all_pass}')

## 2. MoodExtractor — Gemini Call 1 (live)

In [ ]:
from backend.agent.mood_extractor import MoodExtractor

extractor = MoodExtractor()

inputs_to_test = [
    'I feel weird',
    'idk',
    '🥺',
    'burnt out',
    'like Dark but easier',
]

for user_input in inputs_to_test:
    print(f'\nInput: {repr(user_input)}')
    result = extractor.extract(user_input)
    print(f'  Mood     : {result["interpreted_mood"]}')
    print(f'  Intensity: {result["intensity"]}')
    print(f'  Themes   : {result["themes"]}')
    print(f'  Queries  : {result["search_queries"]}')
    print(f'  Confidence: {result["confidence"]}')

## 3. FAISSRetriever — semantic search

In [ ]:
from backend.rag.faiss_retriever import FAISSRetriever

retriever = FAISSRetriever(top_k=10)

queries = ['emotional healing movies', 'bittersweet drama series']
results = retriever.search(queries)

print(f'Retrieved {len(results)} candidates for queries: {queries}\n')
for r in results:
    genres_str = ', '.join(r.get('genres', []))
    print(f"  [{r.get('_retrieval_score', 0):.3f}] {r['title']} ({r.get('year','?')}) — {genres_str}")

## 4. RetrievalEvaluator — quality gate

In [ ]:
from backend.agent.retrieval_evaluator import RetrievalEvaluator

evaluator = RetrievalEvaluator()
report = evaluator.evaluate(results)

print(f'Quality : {report["quality"]}')
print(f'Score   : {report["score"]}')
print(f'Action  : {report["action"]}')
if report['reasons']:
    print(f'Reasons : {report["reasons"]}')

## 5. GeminiGenerator — Gemini Call 2 (live)

In [ ]:
from backend.llm.gemini_generator import GeminiGenerator

mood_data = {
    'interpreted_mood': 'melancholic, reflective',
    'intensity': 'medium',
    'themes': ['loss', 'moving on'],
    'search_queries': queries,
    'confidence': 'high',
}

generator = GeminiGenerator()
recs = generator.generate(mood_data=mood_data, candidates=results)

print(f'Generated {len(recs)} recommendations:\n')
for i, rec in enumerate(recs, 1):
    print(f"{i}. {rec['title']} ({rec.get('year','?')}) — {rec.get('mood_tag','')}")
    print(f"   Platforms: {', '.join(rec.get('platforms', []))}")
    print(f"   {rec.get('explanation', '')}")
    print()

## 6. Full Pipeline — single orchestrated call

In [ ]:
from backend.pipeline.recommender_pipeline import RecommenderPipeline

pipeline = RecommenderPipeline()

test_cases = [
    'I feel melancholic and want something to match that',
    'idk, just something good',
    '🥺',
    'burnt out, need zero effort',
    '',   # should return clarification
]

for user_input in test_cases:
    print(f'\n{"═" * 60}')
    print(f'INPUT: {repr(user_input)}')
    result = pipeline.recommend(user_input)
    print(f'TYPE : {result["type"]}')
    if result['type'] == 'recommendation':
        mood = result.get('interpreted_mood', {})
        print(f'MOOD : {mood.get("interpreted_mood", "N/A")}')
        for rec in result.get('data', []):
            print(f"  🎬 {rec['title']} — {rec.get('mood_tag','')}")
    else:
        print(f'MSG  : {result["follow_up"]}')

## 7. Feedback Refinement Loop

In [ ]:
# First call
result1 = pipeline.recommend('feeling nostalgic and reflective')
sid = result1['session_id']
shown = [r['title'] for r in (result1.get('data') or [])]

print('First recommendations:')
for t in shown:
    print(f'  • {t}')

# Feedback refinement
result2 = pipeline.refine(
    session_id=sid,
    feedback_text='Too dark. Want something uplifting and lighter.',
    shown_titles=shown,
)
shown2 = [r['title'] for r in (result2.get('data') or [])]

print('\nRefined recommendations (should not repeat first set):')
for t in shown2:
    overlap = '⚠ REPEAT' if t in shown else ''
    print(f'  • {t} {overlap}')